In [9]:
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone your repository if it hasn't been cloned yet
    if not os.path.exists('/content/oc-hosp-surge-bn'):
        !git clone https://github.com/AnhJoe/oc-hosp-surge-bn.git
    
    # Change working directory to the project root
    os.chdir('/content/oc-hosp-surge-bn')
    !git pull origin main
    !pip install -q -r requirements.txt

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.25 KiB | 641.00 KiB/s, done.
From https://github.com/AnhJoe/oc-hosp-surge-bn
 * branch            main       -> FETCH_HEAD
   ac1b7ee..beb16ae  main       -> origin/main
Updating ac1b7ee..beb16ae
Fast-forward
 notebooks/01_eda.ipynb | 67 ++++++++++++++++++++++++++++----------------------
 requirements.txt       |  1 -
 2 files changed, 38 insertions(+), 30 deletions(-)


In [10]:
#| output: false
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

root_str = str(root)
if root_str not in sys.path:
    sys.path.insert(0, root_str)
print("Project root:", root)

Project root: /content/oc-hosp-surge-bn


In [ ]:
# Third-party libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Local imports
from src.eda import (
    choose_k_gap_rule,
    evaluate_kmeans_k,
    fit_kmeans,
    pca_project,
    plot_clusters_pca,
    plot_k_diagnostics,
)

# Plot configuration
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "legend.title_fontsize": 9,
        "legend.frameon": True,
        "legend.borderpad": 0.3,
    }
)

In [ ]:
#| output: false

import sys
import zipfile

# Data directories
RAW_DATA_DIR = root / "data" / "raw"
PROCESSED_DATA_DIR = root / "data" / "processed"
SVI_PATH = RAW_DATA_DIR / "SVI_2018_US_county.csv"
SHAPE_PATH = RAW_DATA_DIR / "tl_2018_us_county/tl_2018_us_county.shp"

# Conditionally download data if running in Colab and data is missing
if 'google.colab' in sys.modules:
    if not SVI_PATH.exists() or not SHAPE_PATH.exists():
        print("Running in Colab: Downloading raw data from Google Drive...")
        import gdown
        RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
        
        # Google Drive File ID extracted from the share link
        file_id = '1QiZxOxjk8HeZKvvPEGb9MHQEstDKK55r'
        zip_path = RAW_DATA_DIR / 'raw_data.zip'
        
        # Download using gdown
        gdown.download(id=file_id, output=str(zip_path), quiet=False)
        
        print("Extracting data...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(RAW_DATA_DIR)
        print("Extraction complete.")

# Load svi.csv
svi = pd.read_csv(SVI_PATH)

# Load county lines
counties = gpd.read_file(SHAPE_PATH)